# Relatório 03 — Transformações e Versões da Base para Modelagem

**Projeto:** Previsão de evasão de clientes (churn) em 60 dias

**Etapa:** pré-processamento sem vazamento e preparação das versões A, B e C (Aulas 2 e 3 — PBL / CIn-UFPE)

**Entrada:** `data/processed/base_particionada.csv`

**Saídas:** `versao_a_supervisionada.csv`, `versao_b_mista.csv` e `versao_c_numerica.csv` em `data/processed/`

---

### Resumo

Este relatório executa as transformações que os dois relatórios anteriores deixaram deliberadamente pendentes: o **preenchimento dos valores ausentes**, o **escalonamento das variáveis numéricas** e a **codificação das variáveis categóricas**. O que elas têm em comum é que todas **aprendem parâmetros a partir dos dados** (uma mediana, uma moda, uma média, um desvio padrão, uma lista de categorias). Por isso todas seguem a mesma regra: os parâmetros são
aprendidos (`fit`) **apenas no conjunto de treino** e depois aplicados (`transform`), sem nenhum recálculo, ao treino e ao teste.

Com os dados transformados, o relatório monta três versões da base, cada uma no formato exigido por uma família de algoritmos: a **Versão A**, supervisionada, para os classificadores de churn; a **Versão B**, mista, para o agrupamento com K-prototypes; e a **Versão C**, totalmente numérica,
para o agrupamento com K-means e DBSCAN. As três saem sem nenhum valor ausente, com as mesmas regras de transformação e com a coluna `split` preservada.

## 1. Introdução

### 1.1 Onde estamos no projeto

O Relatório 01 limpou a base com regras fixas e construiu o rótulo `churn`. O Relatório 02 dividiu os 4.976 clientes em treino (3.483) e teste (1.493), de forma estratificada, e usou apenas o treino para uma análise exploratória que terminou em uma lista de recomendações. Este relatório põe essas recomendações em prática:

- imputar a mensalidade **por grupo de fibra óptica**, porque as duas variáveis têm correlação de 0,85 e a mensalidade tem uma distribuição com dois picos;
- criar o indicador `mensalidade_ausente`, porque a falta da mensalidade pode carregar sinal;
- usar imputação simples (moda) nas demais colunas, porque as ausências são pequenas (abaixo de 1%);
- manter todos os registros, inclusive os valores extremos legítimos.

### 1.2 A regra que organiza tudo: `fit` no treino, `transform` em tudo

Toda transformação baseada em dados tem duas fases. Na fase de **ajuste** (`fit`), ela calcula seus parâmetros: o imputador calcula a mediana, o escalador calcula a média e o desvio padrão, o codificador registra a lista de categorias. Na fase de **aplicação** (`transform`), ela usa esses
parâmetros, já fixos, para transformar os dados.

O protocolo de prevenção de vazamento da Aula 2 exige que o ajuste use **somente o treino**. O motivo fica claro quando pensamos no uso real do modelo: em produção, ele receberá clientes novos, um de cada vez, e não haverá como calcular a mediana ou a média "dos clientes de amanhã". As
únicas estatísticas disponíveis serão as que foram aprendidas no passado. O teste simula exatamente essa situação e, para isso, precisa ser transformado com estatísticas que nunca o viram. Se a média usada no escalonamento tivesse sido calculada com treino e teste juntos, uma pequena parte da informação do teste já estaria embutida nas variáveis do treino, e a avaliação final seria otimista. A Aula 3 resume o risco: se estatísticas da base completa forem usadas, o teste apresentará métricas otimistas e falsas.

### 1.3 Por que três versões da base

Algoritmos diferentes exigem formatos diferentes. Um classificador supervisionado precisa do rótulo e aceita qualquer variável numérica. O K-means e o DBSCAN agrupam clientes pela **distância euclidiana**, que só faz sentido entre números na mesma escala. O K-prototypes foi criado justamente para dados mistos: ele usa distância euclidiana nas numéricas e uma medida de
**diferença de categorias** (quantas categorias diferem entre dois clientes) nas categóricas, e por isso precisa receber as categóricas **em texto**, sem one-hot. Em vez de cada integrante do grupo improvisar seu próprio pré-processamento, este notebook gera as três versões a partir das mesmas transformações, garantindo comparabilidade entre os experimentos.

| Versão | Uso | Numéricas | Categóricas | Binárias | `mensalidade_ausente` | `churn` |
| --- | --- | --- | --- | --- | --- | --- |
| **A** — supervisionada | classificação (MP2) | escaladas | one-hot | 0/1 | sim | **alvo** |
| **B** — mista | K-prototypes (MP3) | escaladas | texto | 0/1 (tratadas como categóricas) | não | só referência |
| **C** — numérica | K-means e DBSCAN (MP3) | escaladas | one-hot | 0/1 | não | só referência |

### 1.4 Roteiro do relatório

A **Seção 2** prepara o ambiente, carrega a base particionada e separa treino e teste. A **Seção 3** trata os valores ausentes. A **Seção 4** escalona as variáveis numéricas. A **Seção 5** codifica as categóricas. A **Seção 6** monta e valida as três versões. A **Seção 7** exporta os arquivos, e a **Seção 8** conclui com as orientações de uso de cada versão.

## 2. Ambiente e carregamento

A primeira célula deixa o notebook executável no Colab ou localmente, como nos relatórios anteriores, e importa as bibliotecas. Do scikit-learn usamos dois transformadores: o `StandardScaler`, para o escalonamento, e o `OneHotEncoder`, para a codificação. A imputação é feita diretamente com pandas, porque a mediana **por grupo de fibra** não existe pronta no
`SimpleImputer` e, escrita à mão, fica mais transparente.

In [1]:
# Setup: rode sempre primeiro
import os, sys

# No Colab: clona o repositório (só na primeira vez), entra na pasta, atualiza e instala as
# dependências. Localmente, a partir de notebooks/: sobe para a raiz, onde está o config.py.
if "google.colab" in sys.modules:
    if not os.path.exists("/content/previsao_de_evasao_de_clientes"):
        !git clone -q https://github.com/Gagdm/previsao_de_evasao_de_clientes.git /content/previsao_de_evasao_de_clientes
    %cd /content/previsao_de_evasao_de_clientes
    !git pull -q
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

import numpy as np
import pandas as pd
from IPython.display import display                      # exibe tabelas dentro de laços
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import config   # caminhos, nomes de colunas e parâmetros compartilhados entre os notebooks

/content/previsao_de_evasao_de_clientes


Em seguida, carregamos a base particionada e verificamos se ela é a que o Relatório 02 entregou: colunas esperadas presentes, coluna `split` contendo apenas `treino` e `teste`, e as quantidades de cada conjunto. Depois separamos os dois conjuntos em DataFrames próprios. A partir daqui,
**todo cálculo de estatística usa `treino`**; o `teste` só recebe transformações prontas.

In [2]:
base = pd.read_csv(config.BASE_PARTICIONADA)

# Contrato de entrada: as colunas do projeto e a coluna de partição precisam existir.
colunas_esperadas = [config.ID, *config.NUMERICAS, *config.CATEGORICAS, *config.BINARIAS, config.ALVO, "split"]
faltantes = sorted(set(colunas_esperadas) - set(base.columns))
if faltantes:
    raise ValueError(f"Colunas não encontradas na base particionada: {faltantes}")
# A partição só pode ter os dois rótulos esperados.
assert set(base["split"]) == {"treino", "teste"}, "A coluna split deve conter apenas 'treino' e 'teste'"

# Separa os conjuntos. .copy() cria tabelas independentes, que podem ser alteradas sem afetar 'base'.
treino = base[base["split"] == "treino"].copy()
teste = base[base["split"] == "teste"].copy()

print(f"Base: {len(base)} | Treino: {len(treino)} | Teste: {len(teste)}")
print(f"Churn — treino: {treino[config.ALVO].mean():.2%} | teste: {teste[config.ALVO].mean():.2%}")

# Ausentes por coluna em cada conjunto: é isto que a Seção 3 vai resolver.
display(pd.DataFrame({"ausentes_treino": treino.isna().sum(),
                      "ausentes_teste": teste.isna().sum()}).query("ausentes_treino > 0 or ausentes_teste > 0"))

Base: 4976 | Treino: 3483 | Teste: 1493
Churn — treino: 35.43% | teste: 35.43%


,ausentes_treino,ausentes_teste
mensalidade_rs,94,38
atrasos_6m,31,12
tipo_contrato,17,8
forma_pagamento,14,6


A base chega como esperado: 3.483 clientes de treino e 1.493 de teste, com 35,43% de churn em ambos. As ausências estão nas mesmas quatro colunas apontadas pelo Relatório 02, tanto no treino quanto no teste. As do teste também precisam ser preenchidas, mas, como veremos, com valores calculados exclusivamente no treino.

## 3. Tratamento dos valores ausentes

### 3.1 Estratégias e escolhas

A Aula 2 apresenta quatro alternativas para lidar com ausências, cada uma com um custo:

- **Excluir os registros** é simples, mas reduz a amostra e cria viés de seleção se a falta não for completamente aleatória (MCAR). Aqui, perderíamos cerca de 4% dos clientes, e em produção não seria possível "excluir" um cliente sem mensalidade: o modelo precisaria responder mesmo assim.
- **Imputação simples** (mediana para numéricas, moda para categóricas) preserva todos os registros e é estável. Seu custo é reduzir artificialmente a variância, já que vários clientes recebem o mesmo valor; com ausências pequenas, esse efeito é desprezível.
- **Imputação por modelo** (KNNImputer, árvores) usa as outras colunas para estimar o valor faltante e preserva melhor as correlações, ao preço de mais complexidade.
- **Indicador de ausência**: uma coluna binária que registra se o valor original faltava. Não substitui a imputação; complementa-a, preservando o sinal caso a ausência seja informativa (MNAR).

As escolhas para cada coluna, todas justificadas pelos achados do Relatório 02, são:

| Coluna | Ausentes (treino) | Estratégia | Justificativa |
| --- | --- | --- | --- |
| `mensalidade_rs` | 94 (2,7%) | **mediana por grupo de fibra** + indicador `mensalidade_ausente` | correlação de 0,85 com fibra; distribuição bimodal; churn de 40,4% entre os ausentes |
| `atrasos_6m` | 31 (0,9%) | moda | contagem inteira e pequena; a moda mantém um valor que existe na prática |
| `tipo_contrato` | 17 (0,5%) | moda | categórica com poucas ausências |
| `forma_pagamento` | 14 (0,4%) | moda | categórica com poucas ausências |

A imputação da mensalidade por grupo é uma forma simples de imputação condicional, que fica entre a mediana global e a imputação por modelo. Ela explora a variável que mais explica a mensalidade sem a complexidade de um KNN. Para `atrasos_6m`, a moda é preferível à média porque a média
poderia gerar valores fracionários (como 1,04 atraso), que não existem.

### 3.2 Indicador de ausência

O indicador precisa ser criado **antes** da imputação, enquanto ainda sabemos quais valores faltavam. Ele não usa nenhuma estatística da base, apenas registra um fato de cada cliente, por isso pode ser criado nos dois conjuntos da mesma forma, sem risco de vazamento.

In [3]:
# 1 se a mensalidade estava vazia, 0 caso contrário. .astype(int) converte True/False em 1/0.
for conjunto in (treino, teste):
    conjunto["mensalidade_ausente"] = conjunto["mensalidade_rs"].isna().astype(int)

print(f"mensalidade_ausente = 1 — treino: {treino['mensalidade_ausente'].sum()} | teste: {teste['mensalidade_ausente'].sum()}")

mensalidade_ausente = 1 — treino: 94 | teste: 38


### 3.3 Ajuste da imputação no treino (`fit`)

Agora calculamos, **somente no treino**, os valores que serão usados para preencher as lacunas: a mediana da mensalidade em cada grupo de fibra e a moda das outras três colunas. Guardamos esses valores em variáveis próprias (`medianas_mensalidade` e `modas`). Elas são o equivalente ao "modelo" do imputador: em produção, seriam salvas e reutilizadas para cada cliente novo.

Para efeito de comparação, também calculamos a mediana global, que seria usada por um imputador simples.

In [4]:
# --- FIT: estatísticas calculadas exclusivamente no treino ---

# Mediana da mensalidade dentro de cada grupo de fibra (0 = sem fibra, 1 = com fibra).
# groupby ignora os NaN ao calcular a mediana.
medianas_mensalidade = treino.groupby("fibra_optica")["mensalidade_rs"].median()

# Moda (valor mais frequente) das demais colunas com ausentes. .mode() pode devolver mais de
# um valor em caso de empate; [0] pega o primeiro, para a regra ser determinística.
colunas_moda = ["atrasos_6m", *config.CATEGORICAS]
modas = {coluna: treino[coluna].mode()[0] for coluna in colunas_moda}

print("Mediana da mensalidade por fibra (treino):")
display(medianas_mensalidade.rename("mediana").to_frame().round(2))
print(f"Mediana global da mensalidade (só para comparação): {treino['mensalidade_rs'].median():.2f}")
print(f"Modas (treino): {modas}")

Mediana da mensalidade por fibra (treino):


,mediana
fibra_optica,
0,65.44
1,97.03


Mediana global da mensalidade (só para comparação): 86.04
Modas (treino): {'atrasos_6m': np.float64(1.0), 'tipo_contrato': 'Mensal', 'forma_pagamento': 'Boleto'}


As medianas por grupo mostram por que a imputação condicional faz diferença. A mediana global seria R\$ 86,04, um valor que fica **entre** os dois picos da distribuição e não representa bem nenhum dos grupos: estaria R\$ 20,60 acima do cliente típico sem fibra (R\$ 65,44) e R\$ 10,99 abaixo do cliente típico com fibra (R\$ 97,03). Imputando por grupo, cada cliente recebe um valor coerente com o seu perfil de serviço.

As modas são as esperadas pela análise exploratória: 1 atraso, contrato `Mensal` e pagamento por `Boleto`.

### 3.4 Aplicação da imputação (`transform`)

A mesma função de preenchimento é aplicada ao treino e ao teste, sempre com os valores calculados na etapa anterior. Ao final, verificamos que não sobrou nenhuma ausência nas colunas de atributos.

In [5]:
def imputar(df):
    # Aplica a imputação com os parâmetros aprendidos no treino. Não calcula nada novo.
    df = df.copy()
    # Mensalidade: para cada cliente, busca a mediana do seu grupo de fibra (.map) e usa esse
    # valor apenas onde a mensalidade está vazia (.fillna).
    df["mensalidade_rs"] = df["mensalidade_rs"].fillna(df["fibra_optica"].map(medianas_mensalidade))
    # Demais colunas: preenche com a moda do treino. fillna aceita um dicionário {coluna: valor}.
    df = df.fillna(modas)
    return df

# --- TRANSFORM: aplica as mesmas regras nos dois conjuntos ---
treino_imp = imputar(treino)
teste_imp = imputar(teste)

# Verificação: nenhuma coluna de atributo pode ter ficado com ausentes.
colunas_atributos = [*config.NUMERICAS, *config.CATEGORICAS, *config.BINARIAS]
for nome, conjunto in [("treino", treino_imp), ("teste", teste_imp)]:
    restantes = int(conjunto[colunas_atributos].isna().sum().sum())
    assert restantes == 0, f"Ainda há {restantes} ausentes no {nome}"
print("Nenhum valor ausente nos atributos de treino e teste.")

# Efeito da imputação na distribuição da mensalidade (treino), antes e depois.
display(pd.DataFrame({
    "antes (só preenchidos)": treino["mensalidade_rs"].describe(),
    "depois da imputação": treino_imp["mensalidade_rs"].describe(),
}).round(2))

Nenhum valor ausente nos atributos de treino e teste.


,antes (só preenchidos),depois da imputação
count,3389.00,3483.00
mean,83.42,83.45
std,18.30,18.23
min,49.90,49.90
25%,67.14,66.88
50%,86.04,86.18
75%,98.42,98.16
max,136.42,136.42


Todas as lacunas foram preenchidas, inclusive as 38 mensalidades, 12 atrasos e 14 categorias faltantes do teste, usando apenas estatísticas do treino. A comparação antes/depois mostra que a imputação quase não altera a distribuição da mensalidade: a média e os quartis praticamente não
se movem, e o desvio padrão diminui só levemente. É o efeito esperado da imputação simples sobre uma ausência pequena: ela reduz um pouco a variância, mas sem distorcer o formato da variável.

## 4. Escalonamento das variáveis numéricas

### 4.1 Por que escalonar

As quatro variáveis numéricas têm escalas muito diferentes: o tempo de casa vai de 2 a 72 meses, a mensalidade de R\$ 50 a R\$ 136, os chamados de 0 a 10 e os atrasos de 0 a 5. Para algoritmos baseados em **distância** (K-means, DBSCAN, K-prototypes, KNN), isso é um problema sério: a distância entre dois clientes seria dominada pela variável de maior amplitude. Uma diferença de
30 reais na mensalidade pesaria muito mais que uma diferença de 4 chamados, mesmo que os chamados sejam mais importantes para o churn. Para modelos **lineares** com regularização (como a regressão logística), a escala também afeta quanto cada coeficiente é penalizado. Já as **árvores de decisão** não são afetadas, porque cada divisão olha uma variável de cada vez e só depende da ordem dos valores.

### 4.2 Padronização (z-score)

O `StandardScaler` aplica a padronização:

$$z = \frac{x - \mu}{\sigma}$$

em que μ e σ são a média e o desvio padrão da variável **no treino**. Depois da transformação, cada variável do treino tem média 0 e desvio padrão 1, e o valor z passa a significar "quantos desvios padrão o cliente está acima ou abaixo do cliente médio". Assim, todas as variáveis contribuem em pé de igualdade para a distância.

Uma alternativa seria o `RobustScaler`, que usa mediana e intervalo interquartil e é menos sensível a valores extremos. Como o Relatório 02 mostrou que os extremos que restaram são legítimos e pouco numerosos (até 4,4%), a padronização usual é adequada.

In [6]:
# --- FIT: o escalador aprende média e desvio padrão de cada numérica APENAS no treino ---
escalador = StandardScaler()
escalador.fit(treino_imp[config.NUMERICAS])

# Parâmetros aprendidos: mean_ (média) e scale_ (desvio padrão) de cada coluna.
display(pd.DataFrame({"média (μ)": escalador.mean_, "desvio padrão (σ)": escalador.scale_},
                     index=config.NUMERICAS).round(3))

# --- TRANSFORM: aplica z = (x - μ) / σ com os mesmos μ e σ nos dois conjuntos ---
treino_esc = treino_imp.copy()
teste_esc = teste_imp.copy()
treino_esc[config.NUMERICAS] = escalador.transform(treino_imp[config.NUMERICAS])
teste_esc[config.NUMERICAS] = escalador.transform(teste_imp[config.NUMERICAS])

# Conferência: média e desvio padrão depois da transformação em cada conjunto.
# ddof=0 usa a mesma fórmula de desvio padrão do StandardScaler.
display(pd.DataFrame({
    "média treino": treino_esc[config.NUMERICAS].mean(),
    "desvio treino": treino_esc[config.NUMERICAS].std(ddof=0),
    "média teste": teste_esc[config.NUMERICAS].mean(),
    "desvio teste": teste_esc[config.NUMERICAS].std(ddof=0),
}).round(3))

,média (μ),desvio padrão (σ)
tempo_cliente_meses,24.823,15.499
mensalidade_rs,83.455,18.230
chamados_totais,1.711,1.442
atrasos_6m,1.040,0.950


,média treino,desvio treino,média teste,desvio teste
tempo_cliente_meses,-0.0,1.0,-0.008,1.000
mensalidade_rs,0.0,1.0,-0.010,1.015
chamados_totais,0.0,1.0,-0.008,0.974
atrasos_6m,0.0,1.0,0.018,1.004


No treino, todas as variáveis ficaram com média 0 e desvio padrão 1, exatamente como a fórmula garante. No teste, os valores ficam **próximos, mas não idênticos** a 0 e 1. Essa pequena diferença não é um erro: é a prova de que o teste foi transformado com a média e o desvio do treino, e não com os seus próprios. Se o teste tivesse média exatamente 0 e desvio exatamente 1,
isso indicaria que o escalador foi ajustado nele, ou seja, vazamento.

## 5. Codificação das variáveis categóricas (one-hot)

### 5.1 Fundamentos

Algoritmos numéricos não sabem operar com textos como `Mensal` ou `Pix`. A solução mais ingênua seria numerar as categorias (Mensal = 0, Anual = 1, Bienal = 2), mas isso cria uma **ordem e uma distância falsas**: o algoritmo concluiria que Bienal está "duas vezes mais longe" de Mensal do que Anual, e com formas de pagamento a ordem não teria nenhum significado.

A **codificação one-hot** resolve o problema criando uma coluna binária para cada categoria. Um cliente com contrato anual recebe 1 na coluna `tipo_contrato_Anual` e 0 nas colunas `tipo_contrato_Mensal` e `tipo_contrato_Bienal`. Todas as categorias ficam à mesma distância umas
das outras, sem ordem implícita.

Duas decisões de configuração:

- **`handle_unknown="ignore"`**: se aparecer em produção uma categoria que não existia no treino, o cliente recebe 0 em todas as colunas daquela variável, em vez de o pipeline quebrar. É a recomendação da Aula 3 para o one-hot isolado.
- **Todas as categorias são mantidas** (sem `drop="first"`). Eliminar uma coluna por variável evita a colinearidade perfeita em regressões lineares sem regularização (a chamada "armadilha das dummies"), mas tornaria a codificação assimétrica e prejudicaria a interpretação dos agrupamentos da Versão C, em que todas as categorias precisam estar representadas. Os modelos lineares do scikit-learn usam regularização por padrão, o que já contorna o problema.

Assim como os outros transformadores, o codificador aprende a lista de categorias **somente no treino**.

In [7]:
# --- FIT: o codificador registra as categorias existentes no treino ---
# sparse_output=False devolve uma matriz comum (densa), mais fácil de transformar em DataFrame.
codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
codificador.fit(treino_esc[config.CATEGORICAS])

# Nomes das novas colunas, no formato <variável>_<categoria>.
colunas_onehot = list(codificador.get_feature_names_out(config.CATEGORICAS))
print("Categorias aprendidas no treino:")
for coluna, categorias in zip(config.CATEGORICAS, codificador.categories_):
    print(f"  {coluna}: {list(categorias)}")

def codificar(df):
    # Aplica o one-hot aprendido e devolve uma tabela com as colunas binárias, alinhada ao índice
    # original (para poder ser concatenada às demais colunas do mesmo conjunto).
    matriz = codificador.transform(df[config.CATEGORICAS])
    return pd.DataFrame(matriz.astype(int), columns=colunas_onehot, index=df.index)

# --- TRANSFORM ---
onehot_treino = codificar(treino_esc)
onehot_teste = codificar(teste_esc)

# Conferência: cada cliente deve ter exatamente um "1" por variável categórica.
for coluna in config.CATEGORICAS:
    grupo = [c for c in colunas_onehot if c.startswith(coluna + "_")]
    assert (onehot_treino[grupo].sum(axis=1) == 1).all() and (onehot_teste[grupo].sum(axis=1) == 1).all()

print(f"\nColunas criadas: {colunas_onehot}")
display(onehot_treino.head())

Categorias aprendidas no treino:
  tipo_contrato: ['Anual', 'Bienal', 'Mensal']
  forma_pagamento: ['Boleto', 'Cartão', 'Pix']

Colunas criadas: ['tipo_contrato_Anual', 'tipo_contrato_Bienal', 'tipo_contrato_Mensal', 'forma_pagamento_Boleto', 'forma_pagamento_Cartão', 'forma_pagamento_Pix']


,tipo_contrato_Anual,tipo_contrato_Bienal,tipo_contrato_Mensal,forma_pagamento_Boleto,forma_pagamento_Cartão,forma_pagamento_Pix
0,0,1,0,1,0,0
2,1,0,0,1,0,0
7,0,1,0,1,0,0
8,1,0,0,1,0,0
9,0,0,1,0,0,1


O codificador aprendeu três categorias para cada variável e criou seis colunas binárias. A verificação confirma que todo cliente, no treino e no teste, tem exatamente uma categoria ativa por variável. Isso só é possível porque a imputação da Seção 3 já eliminou os ausentes; caso contrário, os clientes sem categoria ficariam com zeros em todas as colunas.

## 6. Montagem das três versões

Com as transformações prontas, cada versão é montada juntando as colunas adequadas. Treino e teste são montados separadamente com a mesma função e depois concatenados, preservando a coluna `split`. Todas as versões começam com `id_cliente` e `split`, que são **colunas de controle**, e não atributos: nenhuma delas deve entrar em um algoritmo. O identificador, em particular, poderia ser "decorado" pelo modelo sem nenhum valor preditivo.

### 6.1 Versão A — supervisionada

Destinada aos classificadores de churn. Reúne as numéricas escaladas, o one-hot das categóricas, as binárias, o indicador `mensalidade_ausente` e, por último, o alvo `churn`. É a única versão em que `churn` tem papel de variável resposta.

O indicador de ausência entra apenas aqui porque seu propósito é preditivo: permitir que o classificador aproveite um possível sinal da falta de mensalidade. Nos agrupamentos, que buscam perfis de comportamento, ele descreveria uma característica do processo de coleta, e não do cliente, e por ser raro (2,7%) poderia criar grupos artificiais.

In [8]:
colunas_controle = [config.ID, "split"]

def montar_versao_a(df_esc, onehot):
    # Controle + numéricas escaladas + one-hot + binárias + indicador + alvo.
    return pd.concat([
        df_esc[colunas_controle + config.NUMERICAS],
        onehot,
        df_esc[config.BINARIAS + ["mensalidade_ausente", config.ALVO]],
    ], axis=1)   # axis=1 junta lado a lado (colunas), alinhando pelo índice

versao_a = pd.concat([montar_versao_a(treino_esc, onehot_treino),
                      montar_versao_a(teste_esc, onehot_teste)]).sort_values(config.ID).reset_index(drop=True)

print(f"Versão A: {versao_a.shape[0]} linhas × {versao_a.shape[1]} colunas")
display(versao_a.head())

Versão A: 4976 linhas × 16 colunas


,id_cliente,split,tempo_cliente_meses,mensalidade_rs,chamados_totais,atrasos_6m,tipo_contrato_Anual,tipo_contrato_Bienal,tipo_contrato_Mensal,forma_pagamento_Boleto,forma_pagamento_Cartão,forma_pagamento_Pix,fibra_optica,debito_automatico,mensalidade_ausente,churn
0,C0001,treino,0.269491,-0.849966,0.200451,-1.095123,0,1,0,1,0,0,0,0,0,0
1,C0002,teste,-0.375710,-0.960225,-1.186183,-1.095123,0,0,1,0,1,0,0,0,0,0
2,C0003,treino,-0.440230,0.955867,-0.492866,-0.042318,1,0,0,1,0,0,1,1,0,0
3,C0004,teste,-0.440230,0.622347,0.200451,1.010487,0,0,1,0,1,0,0,1,0,0
4,C0005,teste,1.947015,-1.633298,-0.492866,1.010487,1,0,0,0,0,1,0,0,0,0


### 6.2 Versão B — mista (K-prototypes)

Destinada ao K-prototypes, que combina distância euclidiana nas numéricas com uma medida de **dissimilaridade por correspondência** nas categóricas (soma de quantos atributos categóricos diferem entre dois clientes). Por isso as categóricas seguem **em texto**, imputadas mas sem one-hot. As binárias também entram como atributos categóricos: para elas, "diferem ou não
diferem" é exatamente a comparação adequada.

O `churn` fica no arquivo **apenas como coluna de referência**, para que depois seja possível descrever os grupos encontrados (por exemplo, "o grupo 2 tem 60% de churn"). Ele **não pode ser usado como atributo** do agrupamento: um algoritmo não supervisionado deve descobrir a estrutura dos dados sem conhecer a resposta.

In [9]:
def montar_versao_b(df_esc):
    # Controle + numéricas escaladas + categóricas em texto + binárias + churn (referência).
    return df_esc[colunas_controle + config.NUMERICAS + config.CATEGORICAS + config.BINARIAS + [config.ALVO]]

versao_b = pd.concat([montar_versao_b(treino_esc),
                      montar_versao_b(teste_esc)]).sort_values(config.ID).reset_index(drop=True)

# Posições das colunas categóricas DENTRO dos atributos (sem id, split e churn), que é o formato
# pedido pelo parâmetro "categorical" do KPrototypes.
atributos_b = config.NUMERICAS + config.CATEGORICAS + config.BINARIAS
indices_categoricos_b = [atributos_b.index(c) for c in config.CATEGORICAS + config.BINARIAS]

print(f"Versão B: {versao_b.shape[0]} linhas × {versao_b.shape[1]} colunas")
print(f"Atributos: {atributos_b}")
print(f"Índices categóricos para o KPrototypes: {indices_categoricos_b}")
display(versao_b.head())

Versão B: 4976 linhas × 11 colunas
Atributos: ['tempo_cliente_meses', 'mensalidade_rs', 'chamados_totais', 'atrasos_6m', 'tipo_contrato', 'forma_pagamento', 'fibra_optica', 'debito_automatico']
Índices categóricos para o KPrototypes: [4, 5, 6, 7]


,id_cliente,split,tempo_cliente_meses,mensalidade_rs,chamados_totais,atrasos_6m,tipo_contrato,forma_pagamento,fibra_optica,debito_automatico,churn
0,C0001,treino,0.269491,-0.849966,0.200451,-1.095123,Bienal,Boleto,0,0,0
1,C0002,teste,-0.375710,-0.960225,-1.186183,-1.095123,Mensal,Cartão,0,0,0
2,C0003,treino,-0.440230,0.955867,-0.492866,-0.042318,Anual,Boleto,1,1,0
3,C0004,teste,-0.440230,0.622347,0.200451,1.010487,Mensal,Cartão,0,1,0
4,C0005,teste,1.947015,-1.633298,-0.492866,1.010487,Anual,Pix,0,0,0


### 6.3 Versão C — numérica (K-means e DBSCAN)

Destinada aos algoritmos que operam exclusivamente com distância euclidiana. Todas as colunas de atributos são numéricas: as numéricas escaladas, o one-hot das categóricas e as binárias em 0/1. Assim como na Versão B, o `churn` acompanha o arquivo só como referência.

Um ponto de atenção para quem usar esta versão: as colunas 0/1 não foram padronizadas, e uma coluna binária tem desvio padrão de no máximo 0,5, enquanto as numéricas escaladas têm desvio 1. Na prática, cada variável categórica contribui para a distância com um peso parecido com o de uma
numérica (uma mudança de categoria altera duas colunas one-hot ao mesmo tempo), e as binárias pesam um pouco menos. Se os agrupamentos parecerem dominados por um tipo de variável, esse balanceamento é o primeiro ponto a revisar.

In [10]:
def montar_versao_c(df_esc, onehot):
    # Controle + numéricas escaladas + one-hot + binárias + churn (referência).
    return pd.concat([
        df_esc[colunas_controle + config.NUMERICAS],
        onehot,
        df_esc[config.BINARIAS + [config.ALVO]],
    ], axis=1)

versao_c = pd.concat([montar_versao_c(treino_esc, onehot_treino),
                      montar_versao_c(teste_esc, onehot_teste)]).sort_values(config.ID).reset_index(drop=True)

print(f"Versão C: {versao_c.shape[0]} linhas × {versao_c.shape[1]} colunas")
display(versao_c.head())

Versão C: 4976 linhas × 15 colunas


,id_cliente,split,tempo_cliente_meses,mensalidade_rs,chamados_totais,atrasos_6m,tipo_contrato_Anual,tipo_contrato_Bienal,tipo_contrato_Mensal,forma_pagamento_Boleto,forma_pagamento_Cartão,forma_pagamento_Pix,fibra_optica,debito_automatico,churn
0,C0001,treino,0.269491,-0.849966,0.200451,-1.095123,0,1,0,1,0,0,0,0,0
1,C0002,teste,-0.375710,-0.960225,-1.186183,-1.095123,0,0,1,0,1,0,0,0,0
2,C0003,treino,-0.440230,0.955867,-0.492866,-0.042318,1,0,0,1,0,0,1,1,0
3,C0004,teste,-0.440230,0.622347,0.200451,1.010487,0,0,1,0,1,0,0,1,0
4,C0005,teste,1.947015,-1.633298,-0.492866,1.010487,1,0,0,0,0,1,0,0,0


### 6.4 Validação das três versões

Antes de exportar, verificamos o contrato de saída de cada versão: o número de linhas é o da base de entrada, cada cliente aparece uma vez, não há nenhum valor ausente, a partição de cada cliente é a mesma da base particionada e, nas versões A e C, todas as colunas de atributos são numéricas.

In [11]:
versoes = {"A": versao_a, "B": versao_b, "C": versao_c}
# Partição original de cada cliente, para comparar com a de cada versão.
split_original = base.set_index(config.ID)["split"]

resumo = []
for nome, versao in versoes.items():
    assert len(versao) == len(base), f"Versão {nome}: número de linhas diferente da entrada"
    assert versao[config.ID].is_unique, f"Versão {nome}: id_cliente repetido"
    assert not versao.isna().any().any(), f"Versão {nome}: há valores ausentes"
    # A partição de cada cliente precisa ser exatamente a mesma da base particionada.
    assert (versao.set_index(config.ID)["split"] == split_original.loc[versao[config.ID]]).all()
    atributos = versao.drop(columns=colunas_controle + [config.ALVO])
    resumo.append({
        "versão": nome,
        "linhas": len(versao),
        "colunas": versao.shape[1],
        "atributos": atributos.shape[1],
        "atributos não numéricos": int((~atributos.dtypes.map(pd.api.types.is_numeric_dtype)).sum()),
        "ausentes": int(versao.isna().sum().sum()),
    })

# Versões A e C não podem ter nenhum atributo em texto.
for nome in ("A", "C"):
    assert next(r for r in resumo if r["versão"] == nome)["atributos não numéricos"] == 0

display(pd.DataFrame(resumo))

,versão,linhas,colunas,atributos,atributos não numéricos,ausentes
0,A,4976,16,13,0,0
1,B,4976,11,8,2,0
2,C,4976,15,12,0,0


As três versões passam em todas as verificações: 4.976 clientes, sem repetições, sem ausentes e com a mesma partição da base de entrada. A Versão A tem 13 atributos (4 numéricos, 6 one-hot, 2 binários e o indicador de ausência), a Versão B tem 8 (4 numéricos e 4 categóricos, dos quais 2 em texto) e a Versão C tem 12, todos numéricos.

## 7. Exportação

Os três arquivos são salvos em `data/processed/` e relidos para conferir que a gravação manteve
as dimensões.

In [12]:
config.PROCESSADOS.mkdir(parents=True, exist_ok=True)
caminhos = {"A": config.VERSAO_A, "B": config.VERSAO_B, "C": config.VERSAO_C}

for nome, versao in versoes.items():
    versao.to_csv(caminhos[nome], index=False)        # sem o índice numérico do pandas
    conferencia = pd.read_csv(caminhos[nome])          # relê para conferir
    assert conferencia.shape == versao.shape
    print(f"Versão {nome}: {caminhos[nome].name} — {conferencia.shape[0]} linhas × {conferencia.shape[1]} colunas")

Versão A: versao_a_supervisionada.csv — 4976 linhas × 16 colunas
Versão B: versao_b_mista.csv — 4976 linhas × 11 colunas
Versão C: versao_c_numerica.csv — 4976 linhas × 15 colunas


## 8. Conclusão

Este relatório fechou a preparação dos dados do projeto. As três transformações que dependem de estatísticas, imputação, escalonamento e codificação, foram ajustadas **exclusivamente no treino** e aplicadas sem recálculo ao teste. A conferência do escalonamento, com o teste próximo
mas não idêntico a média 0 e desvio 1, comprova que nenhuma informação do teste participou do ajuste.

As decisões seguiram as evidências da análise exploratória. A mensalidade foi imputada pela mediana do seu grupo de fibra (R\$ 65,44 sem fibra e R\$ 97,03 com fibra), em vez de uma mediana global de R\$ 86,04 que não representaria nenhum dos dois perfis, e o indicador `mensalidade_ausente` preserva o possível sinal da ausência. As demais lacunas, todas abaixo de 1%, foram preenchidas pela moda. As numéricas foram padronizadas para que nenhuma domine as distâncias, e as categóricas foram codificadas sem criar ordens artificiais.

O resultado são três bases consistentes entre si, sem nenhum valor ausente, que permitem comparar os experimentos supervisionados e não supervisionados sobre exatamente os mesmos clientes e as mesmas transformações.

### Orientações para quem usar cada versão

| Versão | Arquivo | Atributos (X) | Não usar como atributo | Observações |
| --- | --- | --- | --- | --- |
| **A** | `versao_a_supervisionada.csv` | todas as colunas exceto `id_cliente`, `split` e `churn` | `id_cliente`, `split` | `churn` é o alvo (y). Treinar e ajustar hiperparâmetros só com `split == "treino"`; avaliar uma única vez no teste. |
| **B** | `versao_b_mista.csv` | numéricas, categóricas em texto e binárias | `id_cliente`, `split`, `churn` | Passar ao `KPrototypes` os índices categóricos `[4, 5, 6, 7]`. Usar `churn` só para descrever os grupos depois. |
| **C** | `versao_c_numerica.csv` | numéricas escaladas, one-hot e binárias | `id_cliente`, `split`, `churn` | Para K-means e DBSCAN. Usar `churn` só para descrever os grupos depois. |

Em todos os casos, se a modelagem precisar de novas transformações aprendidas dos dados (por exemplo, balanceamento de classes ou seleção de variáveis), elas devem ser ajustadas dentro do treino, de preferência dentro de um `Pipeline` do scikit-learn, avaliado com validação cruzada estratificada, mantendo o teste intocado até a avaliação final.

---
### Anexo — baixar os arquivos gerados (Colab)

Depois, suba-os em `data/processed/` pelo GitHub (*Add file → Upload files*).

In [13]:
# Só executa no Colab: baixa as três versões para o computador.
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(config.VERSAO_A))
    files.download(str(config.VERSAO_B))
    files.download(str(config.VERSAO_C))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>